## Wearable Style Dataset Analysis

In [91]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialisation 
df = pd.read_csv("wearable_data.csv")

# Notebook display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Plot styling
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

# Initial data inspection
print("\nData shape is:", df.shape)
print("\nData types are:", df.dtypes)
print("\nAmount of null:", df.isna().sum())
print("\nAmount of duplicated:", df.duplicated().sum())
print("\n Desribe data:", df.describe())

def profile_categoricals(df, max_unique=20):
    """Describe clomuns storing categorical data"""
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    # Printing unique value quantities
    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)
        print(f"\n{'=' * 40}")
        print(f"  {col}  ({n_unique} unique values)")
        print(f"{'=' * 40}")

        # Skipping columns with high cardinality
        if n_unique > max_unique:
            print(f"  [Skipped - too many unique values to be useful]")
        else:
            print(df[col].value_counts(dropna=False))

profile_categoricals(df)
print(df.head(10))



Data shape is: (320, 11)

Data types are: user_id                str
date                   str
timestamp              str
steps              float64
heart_rate_bpm     float64
sleep_hours        float64
sleep_quality          str
exercise_type          str
calories_burned    float64
active_minutes       int64
spo2_percent       float64
dtype: object

Amount of null: user_id              0
date                 0
timestamp            0
steps               18
heart_rate_bpm      12
sleep_hours         19
sleep_quality        0
exercise_type      165
calories_burned     10
active_minutes       0
spo2_percent        11
dtype: int64

Amount of duplicated: 16

 Desribe data:          steps  heart_rate_bpm  sleep_hours  calories_burned  active_minutes  \
count   302.00          308.00       301.00           310.00          320.00   
mean   6978.32           77.34         6.30           281.60           58.95   
std    3493.65           29.49         1.21           134.85           27.72   
m

C:\Users\Sense\AppData\Local\Temp\ipykernel_10956\899774226.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category"]).columns


## Remove Duplicate Rows
The dataset contained 16 exact duplicate rows. These reperesent artifacts of the the wearable's data collection. Since the rows are exact copies, we can safely keep the first occurrence of each and drop the rest. This is a lossless operation.

As all duplicates are exact copies, drop_duplicates() with default settings can be used, no need for subsets.

In [92]:
# Data cleaning

def check_duplicates(data_frame):
    """Show all duplicated rows"""
    dupes = data_frame[data_frame.duplicated(keep=False)].sort_values(["user_id", "timestamp"])
    print("Total duplicated rows:", len(dupes))
    display(dupes.head(10))
    return dupes

# Dropping exact duplicates
check_duplicates(df)
df = df.drop_duplicates().reset_index(drop=True)
check_duplicates(df);

Total duplicated rows: 32


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
59,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
157,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
15,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
259,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
71,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
295,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
247,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
275,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
26,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80
193,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80


Total duplicated rows: 0


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent


## Date-time formatting

Due to the multiple date formats present in "date" and "timestamp", pandas converted both columns to an "object" datatype. The "timestamp" column contains both time and date information, making the "date" column redundant. 

Before dropping the "date" column, I converted it to datetime format, and checked it against the dates in "timestamp". This will reveal if the data is actually duplicated. If so, the column will be dropped, because it's fully derivable from timestamp and duplicated storage invites drift.  

I attempted to use pd.to_datetime to convert the mixed formatted dates, but ambiguous day and month combinations resulted in inconsistent conversion. This was discovered when trying to compare the "date" and "timestamp" columns. To fix this, dates were parsed explicitly via a function.

In [93]:
# Check NaN quantity before date-time parsing
print("Amount of null in timestamp column:", df["timestamp"].isna().sum())
print("Amount of null in date column:", df["date"].isna().sum())

# Parsing timestamp
def parse_timestamp(ts):
    """Parse a timestamp in one of three known formats."""
    if pd.isna(ts):
        return pd.NaT
    ts = str(ts).strip()
    
    # Format 1: ISO - "2024-01-15 14:30:00" (or "2024-01-15 14:30")
    if "-" in ts[:5]:
        return pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce") \
            if len(ts) > 16 else pd.to_datetime(ts, format="%Y-%m-%d %H:%M", errors="coerce")
    
    # Format 2: UK - "15/01/2024 14:30"
    if "/" in ts:
        return pd.to_datetime(ts, format="%d/%m/%Y %H:%M", errors="coerce")
    
    # Format 3: Textual - "Jan 15, 2024 14:30"
    return pd.to_datetime(ts, format="%b %d, %Y %H:%M", errors="coerce")
df["timestamp"] = df["timestamp"].apply(parse_timestamp)

# Parsing date
def parse_date(d):
    """Parse a date in one of three known formats."""
    if pd.isna(d):
        return pd.NaT
    d = str(d).strip()
    
    # Format 1: ISO - "2024-01-15"
    if "-" in d[:5]:
        return pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
    
    # Format 2: UK - "15/01/2024"
    if "/" in d:
        return pd.to_datetime(d, format="%d/%m/%Y", errors="coerce")
    
    # Format 3: Textual - "Jan 15, 2024 14:30"
    return pd.to_datetime(d, format="%b %d, %Y", errors="coerce")
df["date"] = df["date"].apply(parse_date)

# Checking date-time parsing
print("\nAmount of null in parsed timestamp column:", df["timestamp"].isna().sum())
print("Amount of null in parsed date column:", df["date"].isna().sum())
print("\nMin timestamp:", df["timestamp"].min())
print("Max timestamp:", df["timestamp"].max())

# Verify that the date part of timestamp matches the date column
dates_match = (df["timestamp"].dt.normalize() == df["date"]).all()
print(f"\nDate components match: {dates_match}")

if dates_match:
    # Drop the date column if it is an exact match
    df = df.drop(columns="date")
    print("Dropped redundant 'date' column")
else:
    # Find out the number of mismatches and print them for manual inspection
    mismatches = (df["timestamp"].dt.normalize() == df["date"])
    print("Number of mismatches =", (len(df) - mismatches.sum()))
    print(df[["timestamp", "date"]][df["timestamp"].dt.normalize() != df["date"]])

# Check results
df.head()

Amount of null in timestamp column: 0
Amount of null in date column: 0

Amount of null in parsed timestamp column: 0
Amount of null in parsed date column: 0

Min timestamp: 2024-01-01 10:51:00
Max timestamp: 2024-02-29 20:46:00

Date components match: True
Dropped redundant 'date' column


,user_id,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
0,U003,2024-02-16 21:31:00,12855.00,82.90,6.94,Good,Running,500.00,101,97.60
1,U004,2024-01-11 08:53:00,7121.00,0.00,6.26,Good,NaN,320.50,55,97.90
2,U001,2024-01-08 19:36:00,4770.00,83.30,7.41,Fair,NaN,138.40,47,97.10
3,U002,2024-01-06 11:21:00,12956.00,67.20,6.56,Excellent,Yoga,500.10,104,95.70
4,U002,2024-02-08 13:07:00,14454.00,62.70,5.98,Good,Cycling,585.50,127,96.90


## Data type formatting

Data types are seemingly correct, however the "steps" column is a float64 despite containing seemingly no decimal values. Pandas has likely converted it due to int64 not being able to handle the NaN values present seen from initial data inspection. As the step counts are only whole numbers, an integer datatype is more suitable.

In [ ]:
# Re-inspect database datatypes
print("Data types are:", df.dtypes)

# Checking to see if any values in "steps" contains decimal values
for step_count in df["steps"]:
    if pd.isna(step_count):
        continue
    elif step_count % 1 != 0.0:
        print(f"'steps' contains decimal values:", step_count)
        break

# Set steps to an Int64 - it contains NaN data so must be
# Int64 rather than int64
df["steps"] = df["steps"].astype("Int64")
print(f"\n'steps' is now an {df["steps"].dtype} data type")

Data types are: user_id                       str
timestamp          datetime64[us]
steps                     float64
heart_rate_bpm            float64
sleep_hours               float64
sleep_quality                 str
exercise_type                 str
calories_burned           float64
active_minutes              int64
spo2_percent              float64
dtype: object

'steps' is now an Int64 data type


## Impossible Data

Columns containing continous data such as: "steps", "heart_rate_bpm", "sleep_hours", "calories_burned", and "active_minutes" will be analysed to see if any entries contain impossible data (e.g. a heart rate of 1000). 

Firstly, biological boundaries will be set at the most extreme minimum and maximum values recored for each variable. Secondly, any outliers will be handled...

In [ ]:
# A negative step count is impossible, while the upper bound 
# reflects the 24 hour step world record by Jesse Castaneda
steps_bounds = [0, 285000]
# A resting heart rate of below 60 beats per minute (BPM) is 
# classified as bradycardia, with 26 BPM being the official
# lowest ever recorded resting heart rate by Martin Brady. 220 BPM 
# is the standard theoretical limit used when calculating maximum 
# heart rate
heart_rate_bounds = [26, 220]
# It is impossible to sleep less than 0 hours or more than 24
sleep_bounds = [0, 24]
# blah
calories_burned = [0, 0]